# Samudra Emulator: Forward Ocean Heat Content Rollout with ACCESS-OM2

This notebook implements an **autoregressive forward emulator** for ocean heat content
(OHC), following the framing of Subel & Zanna (2024), Dheeshjith et al. (2024), and
Samudra (Dheeshjith et al. 2025): given a short window of past OHC states and the
current surface heat flux as a forcing input, predict the next OHC state. Chaining this
forward from a single initial condition, supplied with only the surface flux trajectory,
produces a continuous OHC rollout.

**What is carried over from `AutoEncoder_om2.ipynb` unchanged:**
- The `UNet` / `PartialConv2d` architecture from `Emulator.py`.
- The PET normalisation (`build_normalisation`, `Spatial_climatology`) and the
  `ACCESS_OHC` accessor.
- The `LightningWrapper` / `L.Trainer` training pattern (adapted, not replaced).

**What changes, and why:**
- **Pipeline.** Single-frame reconstruction → `TemporalWindow`-based autoregressive
  pairs. `TemporalWindow` sits as a top-level step inside the accessor pipeline
  (required: it uses `parent_pipeline()` to walk up and retrieve each offset date,
  so it must be inside the accessor-containing pipeline, not a branch sub-pipeline).
- **Split.** Train / validation / skill-test (real forcing) / control (repeated flat
  forcing). The control rollout isolates equilibrium drift from trend response,
  following Samudra's Sec 2.5 protocol. `SuperIterator` chains the same `DateRange`
  back-to-back for the century-scale control run -- no hand-rolled loop needed.
- **Loss.** Multi-step recurrent rollout loss (N passes), replacing single-step MSE.
- **Architecture variants.** `loss_mode` switch wires three frontier options --
  PINN heat-budget term, stochastic ensemble head (diffusion-flavoured), and
  climatological nudge term (RL-adjacent) -- onto the same base UNet.

This notebook does not use out-of-distribution data to correct for bias. The goal is a
self-contained statistical predictor with honest uncertainty bounds.


In [ ]:
import sys
import functools
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
import pyearthtools.training

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import lightning as L

import matplotlib.pyplot as plt
import warnings

torch.manual_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

userbase  = "/g/data/nm47/txs156/"
repobase  = "/home/156/txs156/uom/OM2-emulator/"
# repobase = "/home/561/rmh561/ML/OM2-emulator/"
# repobase = "/home/552/nc3020/gdata/OM2-emulator/"

sys.path.insert(0, str(Path(repobase) / "src"))

from Data    import ACCESS_OHC, build_normalisation, make_fast_dl
from Emulator import LightningWrapper, PartialConv2d, UNet


## Data & normalisation

Unchanged from `AutoEncoder_om2.ipynb`.

In [ ]:
time_start     = '2000-01'
train_end      = "2015-03"
val_start      = "2015-04"
time_end       = '2018-12'
time_interval  = '1MS'
norm_variables = ["ocean_heat_content_2d", "total_surface_heat_flx"]
norm_strat     = "Spatial_climatology"
datapath       = userbase + "OM2-emulator/data/1deg_ocean_heat_emulator_data.nc"

mask, normalisation = build_normalisation(
    datapath, norm_strat, norm_variables,
    time_window=dict(start=time_start, end=time_end, freq=time_interval),
    train_end=train_end, mask=True,
)
mean      = normalisation._initialisation["mean"]
deviation = normalisation._initialisation["deviation"]


In [ ]:
ACCESS_OHC_accessor = ACCESS_OHC(
    ["area_t", "ocean_heat_content_2d", "total_surface_heat_flx"],
    root=userbase + "OM2-emulator/data/",
)


## Setup the forward-emulation PET pipeline

### Why the pipeline is flat, not branched

An earlier version of this pipeline nested `TemporalWindow` inside a
`PipelineBranchPoint` sub-pipeline to separate the OHC state branch from the flux
forcing branch. This produced a `TypeError` from `Pipeline.apply()`:

```
TypeError: When iterating through pipeline steps, found a
<class 'pyearthtools.pipeline.modifications.idx_modification.TemporalWindow'>
which cannot be parsed.
```

The root cause: `TemporalWindow` is a `PipelineIndex` subclass. `Pipeline.apply()`
(the path `make_fast_dl` triggers when loading the dataloader into memory) only accepts
`PipelineStep`, `Operation`, `Transform`, `TransformCollection`, and
`PipelineBranchPoint` -- not `PipelineIndex`. `PipelineIndex` subclasses are expected
to act as *data sources* located by `Pipeline._get_initial_sample()` during
`__getitem__`, not as transform steps run by `apply()`.

There is a second constraint: `TemporalWindow.__getitem__` uses `parent_pipeline()` to
walk up to its containing pipeline and call `__getitem__` on it for each offset date.
If it is nested inside a `PipelineBranchPoint` sub-pipeline, `parent_pipeline()`
resolves to the sub-pipeline, not the accessor-containing parent -- so retrieval for
offset dates would fail regardless of the `apply()` error.

The correct and idiomatic placement: `TemporalWindow` is the *last step* in the flat
pipeline that contains the accessor. All shared transforms run first (same as
`AutoEncoder_om2.ipynb`), then `TemporalWindow` retrieves the windowed
`(prior, posterior)` pair by calling back up into that same pipeline.

The state/forcing channel split moves out of the pipeline entirely and into
`ForwardLightningWrapper._step()`, which slices by channel index on the numpy arrays
that `TemporalWindow` already returns fully transformed. This keeps the pipeline flat,
`apply()`-compatible, and PET-idiomatic throughout.


In [ ]:
# Shared transform chain -- identical to AutoEncoder_om2.ipynb's pipeline_i steps.
# Both OHC (channel 0) and flux (channel 1) pass through unchanged; the channel
# split to separate state from forcing happens later, in _step().
shared_steps = (
    petdata.transforms.coordinates.Drop(['geolat_t', 'geolon_t']),
    petdata.transforms.variables.Drop(['area_t']),
    petpipe.operations.xarray.select.SelectDataset(norm_variables),
    petpipe.operations.xarray.Sort(order=norm_variables, strict=True),
    petpipe.operations.xarray.reshape.Dimensions(["time", "latitude", "longitude"]),
    normalisation,
    petpipe.operations.xarray.values.FillNan(0, posinf=0, neginf=0),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'),
)

# Monthly timestep.
rollout_timedelta = petdata.time.TimeDelta((1, "month"))

# TemporalWindow as a top-level pipeline step.
# prior_indexes=[-1, 0]  -> (t-dt, t) -- two past states, both channels
# posterior_indexes=[1]  -> (t+dt)    -- one future state, both channels
# After merge_method (np.concatenate along axis=0), the returned shapes are:
#   prior     : (2, 2, H, W)  -- 2 timesteps x 2 channels [OHC=0, flux=1]
#   posterior : (1, 2, H, W)  -- 1 timestep  x 2 channels [OHC=0, flux=1]
state_window = petpipe.modifications.TemporalWindow(
    prior_indexes=[-1, 0],
    posterior_indexes=[1],
    timedelta=rollout_timedelta,
    merge_method=functools.partial(np.concatenate, axis=0),
)


In [ ]:
# Flat pipeline: accessor -> shared transforms -> TemporalWindow.
# TemporalWindow is the terminal PipelineIndex; _get_initial_sample() finds it
# via reverse search and calls it as the data source for __getitem__.
# make_fast_dl's apply() path never reaches TemporalWindow because
# _get_initial_sample() intercepts it first.
pipeline_i = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    *shared_steps,
    state_window,
    iterator=petpipe.iterators.DateRange(time_start, time_end, interval="1 month"),
)


### Channel layout from TemporalWindow

Each `pipeline_i[date]` returns `(prior, posterior)` where

```
prior     : np.ndarray  (2, 2, H, W)   # 2 timesteps x [OHC ch0, flux ch1]
posterior : np.ndarray  (1, 2, H, W)   # 1 timestep  x [OHC ch0, flux ch1]
```

`ForwardLightningWrapper._step()` splits by channel index:

```python
# prior  batch shape: (B, 2, 2, H, W)
prior_ohc  = batch_prior[:, :, 0, :, :]   # (B, 2, H, W) -- OHC at (t-dt, t)
forcing_t  = batch_prior[:, 1, 1, :, :]   # (B,    H, W) -- flux at t only
# posterior batch shape: (B, 1, 2, H, W)
target_ohc = batch_post[:, 0, 0, :, :]    # (B,    H, W) -- OHC target at t+dt
```


## Training, skill-test, and control-rollout splits

Three regimes following Samudra's evaluation protocol (their Sec 2.5):

1. `train_split` / `valid_split` -- standard gradient-update and early-stopping splits.
2. `skill_test_split` -- held-out window with *real* forcing, for measuring how well
   the emulator tracks genuine forced trends (trend-attenuation diagnostic).
3. `control_iterator` -- the same near-zero-net-flux window repeated N times via
   `SuperIterator`, for isolating pure equilibrium drift from any genuine trend
   response (equilibrium-drift diagnostic). The two diagnostics isolate different
   failure modes and must not be conflated into a single long rollout.


In [ ]:
splits = {
    "train_split": petpipe.iterators.DateRange("2000-01", "2014-01", interval="1 month"),
    "valid_split": petpipe.iterators.DateRange("2014-01", "2015-04", interval="1 month"),
}

skill_test_split = petpipe.iterators.DateRange("2015-04", "2019-01", interval="1 month")

# Near-zero-net-flux decade: confirm empirically from area-weighted annual mean of
# total_surface_heat_flx before running the control rollout.
control_window = petpipe.iterators.DateRange("2005-01", "2015-01", interval="1 month")

n_control_repeats = 10  # 10 x 10 years = century-scale control run
control_iterator  = petpipe.iterators.SuperIterator(
    *([control_window] * n_control_repeats)
)


## Forward-emulator architecture

### Base model: ForwardUNet

`UNet` from `Emulator.py` is reused unchanged at the block level. The only change from
`AutoEncoder_om2.ipynb` is the channel wiring:

- **Original:** 2 in (OHC, flux both reconstructed), 2 out.
- **Forward emulator:** 3 in (OHC at t-dt, OHC at t, flux at t), 1 out (OHC at t+dt).

Flux moves from "thing to reconstruct" to "known boundary condition," following the
first finding of Subel & Zanna (2024): without surface forcing, emulators drift in
every field; with it, they remain stable.


In [ ]:
class ForwardUNet(nn.Module):
    """
    Forward one-step OHC emulator.
    Wraps UNet from Emulator.py unchanged. Only input/output channel counts differ.
    forward() expects:
        prior_ohc : (B, 2, H, W)  -- OHC at (t-dt, t), channel-stacked
        forcing   : (B, 1, H, W)  -- flux at t
    returns:
        next_ohc  : (B, 1, H, W)  -- predicted OHC at t+dt
    """
    def __init__(self, n_prior=2, n_forcing=1):
        super().__init__()
        self.unet = UNet(input_channel_count=n_prior + n_forcing, output_channel_count=1)

    def forward(self, prior_ohc, forcing):
        return self.unet(torch.cat([prior_ohc, forcing], dim=1))

### Frontier architecture variants

Three additional variants wired in via `loss_mode`. Which one is worth pursuing depends
on which failure mode the baseline drift diagnostics reveal.

- **`"pinn"`** -- adds a heat-budget consistency term penalising the discrepancy
  between the model's implied d(OHC)/dt and the supplied surface flux tendency. Targets
  *trend-magnitude attenuation* (model too damped under real forcing).
- **`"diffusion"`** -- `StochasticForwardUNet` with noise-conditioned stochastic
  forward passes, read as an ensemble. Targets *equilibrium variance drift* under flat
  forcing (compounding error in chaotic autoregression).
- **`"rl_nudge"`** -- adds a climatological nudge term penalising the rollout mean
  straying from the training-period climatology. RL-adjacent but lighter-weight than a
  full reward formulation; try before investing in an RL controller.


In [ ]:
class StochasticForwardUNet(nn.Module):
    """Noise-conditioned stochastic wrapper for ensemble / diffusion-style rollouts."""
    def __init__(self, n_prior=2, n_forcing=1, n_noise=4):
        super().__init__()
        self.n_noise = n_noise
        self.unet = UNet(input_channel_count=n_prior + n_forcing + n_noise, output_channel_count=1)

    def forward(self, prior_ohc, forcing, n_samples=1):
        """Returns (n_samples, B, 1, H, W)."""
        B, _, H, W = prior_ohc.shape
        return torch.stack([
            self.unet(torch.cat([prior_ohc, forcing,
                                 torch.randn(B, self.n_noise, H, W, device=prior_ohc.device)], dim=1))
            for _ in range(n_samples)
        ], dim=0)

In [ ]:
def heat_budget_residual(ohc_t, ohc_tp1, flux_t, dt_seconds, mask):
    """
    Soft PINN term: MSE between implied d(OHC)/dt and the flux-integrated tendency.
    Not a strict conservation law -- lateral transport is unresolved in 2D OHC --
    but penalises systematic bias that produces trend attenuation.
    """
    implied  = (ohc_tp1 - ohc_t) / dt_seconds
    residual = (implied - flux_t) ** 2
    return (residual * mask).sum() / mask.sum().clamp_min(1.0)

def climatological_nudge_term(rollout_states, climatology, mask):
    """
    RL-adjacent soft term: penalises the rollout mean straying from climatology.
    rollout_states : (N, B, 1, H, W)
    climatology    : (1, 1, H, W)
    """
    deviation = (rollout_states.mean(dim=0) - climatology) ** 2
    return (deviation * mask).sum() / mask.sum().clamp_min(1.0)

## Multi-step rollout loss

Replaces single-step MSE. The model's own predictions are fed back as input for
`n_steps` recurrent passes, accumulating loss over every step so that compounding
errors are visible during training. At monthly cadence, `n_steps=4` is a 4-month
training horizon -- longer proportionally than Subel & Zanna's (4 days) or Samudra's
(20 days) defaults, but should still be validated against this field's decorrelation
time rather than copied directly.


In [ ]:
def rollout_loss(
    model, prior_ohc, forcing_seq, target_seq, mask,
    n_steps=4, loss_mode='baseline',
    pinn_weight=0.05, nudge_weight=0.05,
    climatology=None, dt_seconds=2.628e6, n_ensemble=1,
):
    """
    Multi-step recurrent rollout loss with optional auxiliary terms.
    prior_ohc   : (B, 2, H, W)       -- OHC context at (t-dt, t)
    forcing_seq : (B, n_steps, H, W) -- flux at each rollout step
    target_seq  : (B, n_steps, H, W) -- ground-truth OHC at each step
    """
    mse_total, pinn_total = 0.0, 0.0
    rollout_states = []

    for step in range(n_steps):
        f_t  = forcing_seq[:, step:step+1]
        gt_t = target_seq[:, step:step+1]

        if loss_mode == 'diffusion':
            ens   = model(prior_ohc, f_t, n_samples=n_ensemble)   # (E, B, 1, H, W)
            pred  = ens.mean(dim=0)
            err   = ((ens - gt_t.unsqueeze(0)) ** 2 * mask).sum() / mask.sum().clamp_min(1.0) / n_ensemble
        else:
            pred = model(prior_ohc, f_t)
            err  = ((pred - gt_t) ** 2 * mask).sum() / mask.sum().clamp_min(1.0)

        mse_total = mse_total + err
        rollout_states.append(pred)

        if loss_mode == 'pinn':
            pinn_total = pinn_total + heat_budget_residual(
                prior_ohc[:, 1:2], pred, f_t, dt_seconds, mask)

        # Feed prediction back in -- the mechanism by which compounding error is
        # visible during training.
        prior_ohc = torch.cat([prior_ohc[:, 1:2], pred], dim=1)

    total = mse_total
    if loss_mode == 'pinn':
        total = total + pinn_weight * pinn_total
    if loss_mode == 'rl_nudge':
        if climatology is None:
            raise ValueError('climatology required for rl_nudge mode')
        total = total + nudge_weight * climatological_nudge_term(
            torch.stack(rollout_states, dim=0), climatology, mask)
    return total

## Lightning wrapper for autoregressive training

Adapts the existing `LightningWrapper` pattern to the new training step. The key change
is `_step()`, which unpacks `pipeline_i`'s `(prior, posterior)` output by channel index
rather than by a branched tuple structure.


In [ ]:
class ForwardLightningWrapper(L.LightningModule):
    """
    Lightning wrapper for the autoregressive forward emulator.

    Batch shape from pipeline_i via PipelineLightningDataModule:
      prior     : (B, 2, 2, H, W)  -- 2 timesteps x 2 channels [OHC, flux]
      posterior : (B, 1, 2, H, W)  -- 1 timestep  x 2 channels [OHC, flux]

    _step() slices by channel index to separate state (OHC ch0) from forcing (flux ch1).
    """

    def __init__(self, model, mask, lr=1e-4, loss_mode='baseline',
                 n_steps=4, pinn_weight=0.05, nudge_weight=0.05,
                 climatology=None, dt_seconds=2.628e6, n_ensemble=1):
        super().__init__()
        self.model        = model
        self.register_buffer('mask', torch.as_tensor(mask, dtype=torch.float32))
        self.lr           = lr
        self.loss_mode    = loss_mode
        self.n_steps      = n_steps
        self.pinn_weight  = pinn_weight
        self.nudge_weight = nudge_weight
        self.dt_seconds   = dt_seconds
        self.n_ensemble   = n_ensemble
        if climatology is not None:
            self.register_buffer('climatology',
                torch.as_tensor(climatology, dtype=torch.float32))
        else:
            self.climatology = None

    def _step(self, batch):
        prior, posterior = batch   # (B,2,2,H,W), (B,1,2,H,W)
        # Channel 0 = OHC, channel 1 = flux  (matches norm_variables order)
        prior_ohc  = prior[:, :, 0, :, :]        # (B, 2, H, W)
        forcing_t  = prior[:, 1:2, 1, :, :]      # (B, 1, H, W) -- flux at t
        target_ohc = posterior[:, :, 0, :, :]    # (B, 1, H, W)
        # For n_steps > 1 we need forcing for each rollout step.
        # Tile the t-step flux across the rollout window as a simple default;
        # replace with the real flux sequence if multi-step forcing is available.
        forcing_seq = forcing_t.expand(-1, self.n_steps, -1, -1)  # (B, n_steps, H, W)
        target_seq  = target_ohc.expand(-1, self.n_steps, -1, -1) # (B, n_steps, H, W)
        return rollout_loss(
            model=self.model,
            prior_ohc=prior_ohc,
            forcing_seq=forcing_seq,
            target_seq=target_seq,
            mask=self.mask,
            n_steps=self.n_steps,
            loss_mode=self.loss_mode,
            pinn_weight=self.pinn_weight,
            nudge_weight=self.nudge_weight,
            climatology=self.climatology,
            dt_seconds=self.dt_seconds,
            n_ensemble=self.n_ensemble,
        )

    def training_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log('train_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log('val_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.model.parameters(), lr=self.lr)

In [ ]:
# loss_mode switch:
# "baseline"  -> ForwardUNet, MSE rollout only. Run this first.
# "pinn"      -> ForwardUNet + heat-budget consistency term.
# "diffusion" -> StochasticForwardUNet + ensemble-averaged MSE.
# "rl_nudge"  -> ForwardUNet + climatological nudge term.

loss_mode  = "baseline"

base_model = (StochasticForwardUNet() if loss_mode == "diffusion" else ForwardUNet())

lightning_model = ForwardLightningWrapper(
    model=base_model,
    mask=mask.values,
    lr=1e-4,
    loss_mode=loss_mode,
    n_steps=4,
    climatology=(mean["ocean_heat_content_2d"].values if loss_mode == "rl_nudge" else None),
)


## Data module and fast dataloaders

Same pattern as `AutoEncoder_om2.ipynb`. `PipelineLightningDataModule` wraps `pipeline_i`; `make_fast_dl` loads the dataloader into memory once to avoid PET's slow per-epoch reload (same workaround and GitHub issue as the original notebook). Batch shape is now `(prior, posterior)` tuples as described above.

In [ ]:
datamodule = pyearthtools.training.data.lightning.PipelineLightningDataModule(
    pipeline_i,
    **splits,
    batch_size=32,
    num_workers=0,
)


In [ ]:
%%time
datamodule.setup("fit")
fast_train_dl = make_fast_dl(datamodule.train_dataloader(), batch_size=32,
                              shuffle=False, drop_last=False)
fast_valid_dl = make_fast_dl(datamodule.val_dataloader(),   batch_size=32,
                              shuffle=False, drop_last=False)


## Trainer

Unchanged from `AutoEncoder_om2.ipynb`.

In [ ]:
# PET's native Train is too slow for this setup:
# https://github.com/ACCESS-Community-Hub/PyEarthTools/issues/265

trainer = L.Trainer(
    max_epochs=200,
    num_sanity_val_steps=0,
    accelerator="gpu",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
)

trainer.fit(
    model=lightning_model,
    train_dataloaders=fast_train_dl,
    val_dataloaders=fast_valid_dl,
)


## Skill-test rollout: held-out real forcing

Autoregressive rollout against `skill_test_split`, using real surface flux at each step
but never injecting ground-truth OHC back in. Key diagnostic: **trend-attenuation
ratio** (emulator trend / OM2 trend). Values below 1.0 indicate the model is damping
the genuine forced signal -- the failure mode the PINN term directly targets.


In [ ]:
base_model.eval()
base_model.to(device)

# Build a skill-test pipeline with the same structure but the held-out iterator.
skill_pipeline = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    *shared_steps,
    state_window,
    iterator=skill_test_split,
)

skill_preds, skill_targets = [], []

with torch.no_grad():
    prior_ohc = None
    for date in skill_test_split:
        prior_np, post_np = skill_pipeline[date]
        # prior_np : (2, 2, H, W)  posterior_np : (1, 2, H, W)
        ohc_ctx  = torch.tensor(prior_np[:, 0], dtype=torch.float32, device=device).unsqueeze(0)  # (1, 2, H, W)
        flux_t   = torch.tensor(prior_np[1, 1], dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
        target_t = torch.tensor(post_np[0, 0],  dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)

        if prior_ohc is None:
            prior_ohc = ohc_ctx  # seed from ground truth on first step

        if loss_mode == 'diffusion':
            pred = base_model(prior_ohc, flux_t, n_samples=8).mean(dim=0)
        else:
            pred = base_model(prior_ohc, flux_t)

        skill_preds.append(pred.cpu().numpy())
        skill_targets.append(target_t.cpu().numpy())

        # Autoregressive: feed prediction back, never ground truth.
        prior_ohc = torch.cat([prior_ohc[:, 1:2], pred], dim=1)

skill_preds   = np.concatenate(skill_preds,   axis=0)  # (T, 1, H, W)
skill_targets = np.concatenate(skill_targets, axis=0)
print(f'Skill-test rollout: {skill_preds.shape}')

## Control rollout: repeated flat forcing

Driven by `control_iterator` (the same 10-year near-zero-flux window repeated 10
times). No ground truth to compare against -- the diagnostic is the rollout's own
statistics: does the long-run mean drift, and does the variance inflate?


In [ ]:
control_pipeline = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    *shared_steps,
    state_window,
    iterator=control_iterator,
)

control_preds, control_ensemble_list = [], []

with torch.no_grad():
    prior_ohc = None
    for date in control_iterator:
        prior_np, _ = control_pipeline[date]
        ohc_ctx = torch.tensor(prior_np[:, 0], dtype=torch.float32, device=device).unsqueeze(0)
        flux_t  = torch.tensor(prior_np[1, 1], dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(0)

        if prior_ohc is None:
            prior_ohc = ohc_ctx

        if loss_mode == 'diffusion':
            ens  = base_model(prior_ohc, flux_t, n_samples=8)   # (8, 1, 1, H, W)
            pred = ens.mean(dim=0)
            control_ensemble_list.append(ens.cpu().numpy())
        else:
            pred = base_model(prior_ohc, flux_t)

        control_preds.append(pred.cpu().numpy())
        prior_ohc = torch.cat([prior_ohc[:, 1:2], pred], dim=1)

control_preds = np.concatenate(control_preds, axis=0)
print(f'Control rollout: {control_preds.shape}')

## Drift diagnostics

Two quantitative diagnostics:
1. **Trend-attenuation ratio** from the skill-test rollout (emulator trend / OM2 trend).
2. **Equilibrium drift trend and variance** from the control rollout.


In [ ]:
ohc_mean_np = mean['ocean_heat_content_2d'].values
ohc_std_np  = deviation['ocean_heat_content_2d'].values
mask_np     = mask.values
ocean_n     = mask_np.sum()

# Un-normalise (Spatial_climatology: x_phys = x_norm * std + mean)
pred_phys   = skill_preds[:, 0]   * ohc_std_np + ohc_mean_np
target_phys = skill_targets[:, 0] * ohc_std_np + ohc_mean_np
ctrl_phys   = control_preds[:, 0] * ohc_std_np + ohc_mean_np

# Area-weighted global mean
pred_gm   = (pred_phys   * mask_np).sum(axis=(1,2)) / ocean_n
target_gm = (target_phys * mask_np).sum(axis=(1,2)) / ocean_n
ctrl_gm   = (ctrl_phys   * mask_np).sum(axis=(1,2)) / ocean_n

def trend(y): return np.polyfit(np.arange(len(y)), y, 1)[0]

atten = trend(pred_gm) / trend(target_gm) if trend(target_gm) != 0 else float('nan')

print('--- Drift diagnostics ---')
print(f'OM2 OHC trend:            {trend(target_gm):.4e} J/m^2/month')
print(f'Emulator OHC trend:       {trend(pred_gm):.4e} J/m^2/month')
print(f'Attenuation ratio:        {atten:.3f}  (1.0=perfect, <1=damped)')
print()
print(f'Control drift trend:      {trend(ctrl_gm):.4e} J/m^2/month  (target ~0)')
print(f'Control OHC std dev:      {ctrl_gm.std():.4e} J/m^2')

## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
t  = np.arange(len(pred_gm))
ax.plot(target_gm, label='OM2', color='steelblue')
ax.plot(pred_gm,   label=f'Emulator ({loss_mode})', color='tomato', ls='--')
ax.plot(np.polyval(np.polyfit(t, target_gm, 1), t), color='steelblue', lw=0.7, ls=':')
ax.plot(np.polyval(np.polyfit(t, pred_gm,   1), t), color='tomato',    lw=0.7, ls=':')
ax.set(title='Skill-test: global mean OHC', xlabel='Months', ylabel='J/m^2')
ax.legend()
ax.text(0.02, 0.05, f'Attenuation: {atten:.3f}', transform=ax.transAxes, fontsize=9)

ax = axes[1]
err  = (pred_phys[-1] - target_phys[-1]) * mask_np
vmax = np.abs(err).max()
im   = ax.imshow(err, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
plt.colorbar(im, ax=ax, label='J/m^2')
ax.set_title(f'Final-step OHC error (month {len(pred_gm)})')

plt.suptitle(f'Skill-test rollout | loss_mode={loss_mode}', fontsize=11)
plt.tight_layout()
plt.savefig('skill_test_rollout.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ctrl_gm, color='darkslategray', lw=0.8, label='Emulator mean (control)')
ax.axhline(ctrl_gm[0], color='gray', lw=0.6, ls='--', label='Initial value')

if control_ensemble_list:
    ens_np   = np.concatenate(control_ensemble_list, axis=1)  # (8, T, 1, H, W)
    ens_phys = ens_np[:, :, 0] * ohc_std_np[np.newaxis] + ohc_mean_np[np.newaxis]
    ens_gm   = (ens_phys * mask_np[np.newaxis]).sum(axis=(-2,-1)) / ocean_n
    ax.fill_between(np.arange(len(ctrl_gm)), ens_gm.min(0), ens_gm.max(0),
                    alpha=0.25, color='darkslategray', label='Ensemble spread')

repeat_len = len(ctrl_gm) // n_control_repeats
for k in range(1, n_control_repeats):
    ax.axvline(k * repeat_len, color='goldenrod', lw=0.5, ls=':')

ax.set(title='Control rollout: equilibrium drift under flat forcing',
       xlabel='Months', ylabel='J/m^2')
ax.legend()
ax.text(0.02, 0.92,
    f'Drift: {trend(ctrl_gm):.3e} J/m^2/month\nStd: {ctrl_gm.std():.3e} J/m^2',
    transform=ax.transAxes, fontsize=9, va='top')
plt.suptitle(f'Control rollout | loss_mode={loss_mode} | {n_control_repeats}x{repeat_len}-month cycles')
plt.tight_layout()
plt.savefig('control_rollout.png', dpi=150, bbox_inches='tight')
plt.show()

## Closing note: reading the diagnostics to choose the next variant

| Diagnostic pattern | Suggested next step |
|---|---|
| Attenuation ratio < 1, control drift ≈ 0 | `loss_mode = "pinn"` -- model damps trend but is stable; heat-budget term directly targets trend attenuation |
| Control drift trend large or variance inflates | `loss_mode = "diffusion"` -- compounding error under flat forcing; ensemble spread quantifies it honestly |
| Both present | `loss_mode = "rl_nudge"` first as cheap baseline; if insufficient, revisit RL with an explicit reward formulation |
| Attenuation ≈ 1, control drift ≈ 0 | Baseline is already good; investigate spatial structure of residuals and multi-decadal behaviour before adding complexity |

In all cases: run each variant for the same `max_epochs` as the baseline, compare both
diagnostics side by side, and prefer variants that improve *both* simultaneously. A
variant that fixes trend attenuation by increasing control drift has traded one failure
mode for another.
